# optimizer-state-tensor-buffers composite — cx5: momentum SGD: velocity buffer feeds the in-place param update

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `optimizer-state-tensor-buffers`, `inplace-param-update`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "optimizer-state-tensor-buffers"
DD_ATOM_IDS = ["optimizer-state-tensor-buffers", "inplace-param-update"]
DD_SUBTOPICS = ["Optimizer: Per-param state buffers", "PyTorch: In-place param update"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Momentum SGD has TWO in-place updates per step, in this order:
1. Update the velocity buffer (state): `v ← momentum * v + grad`.
2. Update the parameter (in place): `p.data -= lr * v`.

The composition is: the JUST-UPDATED velocity (atom A: optimizer-state-tensor-buffers) is the quantity that feeds the in-place param update (atom B: inplace-param-update). If you use the OLD velocity in the param step, you've implemented something different — closer to plain SGD with a stale velocity term.

PyTorch's `torch.optim.SGD` with `momentum > 0` follows this exact order: velocity first, then param. The reference update rule (without dampening, nesterov, or weight_decay) is:
```
v_t = momentum * v_{t-1} + g_t
p_t = p_{t-1} - lr * v_t                # uses v_t, not v_{t-1}.
```

**Idiomatic in-place version (what we do here).**
```python
@t.no_grad()
def step(self):
    for p, v in zip(self.params, self.velocities):
        if p.grad is None: continue
        v.mul_(self.momentum).add_(p.grad)   # atom A: update buffer in place.
        p.data.add_(v, alpha=-self.lr)       # atom B: param update in place, uses NEW v.
```
The `add_(v, alpha=-lr)` is the in-place equivalent of `p.data -= lr * v`.

**Why both atoms together.** This is the canonical 'optimizer that needs state' pattern. Same shape applies to Adam, RMSProp, Adagrad — the only thing that changes is how the buffer update is computed.

### Composite Exercise — momentum SGD: velocity buffer feeds the in-place param update

**Atoms exercised together**: `optimizer-state-tensor-buffers`, `inplace-param-update`

Implement `cx5_make_sgdm()` — return `SGDM` with a working momentum step.

- `SGDM(params, lr, momentum=0.9)`:
  - `self.params = list(params)`
  - `self.lr = lr; self.momentum = momentum`
  - `self.velocities = [t.zeros_like(p) for p in self.params]` (atom A).
- `SGDM.step(self)`:
  - For each `(p, v)` in `zip(self.params, self.velocities)`:
    - If `p.grad is None`, skip.
    - Update v IN-PLACE: `v.mul_(self.momentum).add_(p.grad)` (or equivalent `copy_`/`-=`).
    - Update p IN-PLACE: `p.data -= self.lr * v` (or `p.data.add_(v, alpha=-self.lr)`) (atom B).
  - Wrap in `t.no_grad()`.

The test cross-checks against `torch.optim.SGD(..., momentum=0.9)` over 4 steps with varying gradients. It also checks that the velocity buffer was used in the CORRECT order (updated FIRST, then fed into the param step) by re-deriving the expected values by hand.

In [ ]:
def cx5_make_sgdm():
    class SGDM:
        def __init__(self, params, lr, momentum=0.9):
            self.params = list(params)
            self.lr = lr
            self.momentum = momentum
            # Atom A (optimizer-state-tensor-buffers): velocity buffers, zero-init.
            self.velocities = [t.zeros_like(p) for p in self.params]

        @t.no_grad()
        def step(self):
            for p, v in zip(self.params, self.velocities):
                if p.grad is None:
                    continue
                # Update velocity IN PLACE: v = momentum*v + grad.
                # mul_ + add_ is the idiomatic two-step in-place form.
                v.mul_(self.momentum).add_(p.grad)
                # Atom B (inplace-param-update): p.data -= lr * v (using the NEW v).
                p.data.add_(v, alpha=-self.lr)

    return SGDM


<details><summary>Show solution — cx5</summary>

```python
def cx5_make_sgdm():
    class SGDM:
        def __init__(self, params, lr, momentum=0.9):
            self.params = list(params)
            self.lr = lr
            self.momentum = momentum
            # Atom A (optimizer-state-tensor-buffers): velocity buffers, zero-init.
            self.velocities = [t.zeros_like(p) for p in self.params]

        @t.no_grad()
        def step(self):
            for p, v in zip(self.params, self.velocities):
                if p.grad is None:
                    continue
                # Update velocity IN PLACE: v = momentum*v + grad.
                # mul_ + add_ is the idiomatic two-step in-place form.
                v.mul_(self.momentum).add_(p.grad)
                # Atom B (inplace-param-update): p.data -= lr * v (using the NEW v).
                p.data.add_(v, alpha=-self.lr)

    return SGDM
```

The order matters: PyTorch's SGD computes `v_t = mu*v_{t-1} + g_t` FIRST, then `p_t = p_{t-1} - lr*v_t`. If you swap the order (use the OLD velocity in the param step), you get a step that's effectively 'plain SGD with a stale momentum kicker' — convergence looks similar early, then diverges on harder problems. The `add_(v, alpha=-self.lr)` form fuses the multiplication and subtraction into a single CUDA kernel; the equivalent `p.data -= self.lr * v` allocates a temporary `lr*v` tensor. Both are mathematically identical.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx5'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx5',
        'subtopics': ["Optimizer: Per-param state buffers", "PyTorch: In-place param update"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()